preprocessing process:
- we define one schema for all clases (text, label)
- don't join them still
- preprocess each dataset separately (!!!CAREFULT)
- merge into one final dataset
- 

## Preprocessing News

Based on the eda, the news dataset is pretty clean, so the preprocessing should be light and we should make sure that we don't over clean it.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
NEWS_DIR = Path("../data/raw/news/bbc")

data = []

for category_dir in NEWS_DIR.iterdir():
    if category_dir.is_dir():
        label = category_dir.name
        
        for file in category_dir.glob("*.txt"):
            text = file.read_text(encoding="latin-1")
            data.append({
                "label": label,
                "text": text,
                "filename": file.name
            })

news = pd.DataFrame(data)
news.shape

(2225, 3)

In [3]:
news.head()

,label,text,filename
0,entertainment,Musicians to tackle US red tape\n\nMusicians' ...,289.txt
1,entertainment,"U2's desire to be number one\n\nU2, who have w...",262.txt
2,entertainment,Rocker Doherty in on-stage fight\n\nRock singe...,276.txt
3,entertainment,Snicket tops US box office chart\n\nThe film a...,060.txt
4,entertainment,Ocean's Twelve raids box office\n\nOcean's Twe...,074.txt


In [4]:
# make working copy
news_prep = news.copy()
news_prep.shape

(2225, 3)

In [5]:
# keep only relevant columns which are: docuemnt text, class label and maybe filename for reference
news_prep = news_prep[["filename", "label", "text"]].copy()
news_prep.head()

,filename,label,text
0,289.txt,entertainment,Musicians to tackle US red tape\n\nMusicians' ...
1,262.txt,entertainment,"U2's desire to be number one\n\nU2, who have w..."
2,276.txt,entertainment,Rocker Doherty in on-stage fight\n\nRock singe...
3,060.txt,entertainment,Snicket tops US box office chart\n\nThe film a...
4,074.txt,entertainment,Ocean's Twelve raids box office\n\nOcean's Twe...


In [6]:
# standardize text column just in case, even though eda showed no missing values, it ensures that all rows are strings
news_prep["text"] = news_prep["text"].fillna("").astype(str)

In [7]:
# strip leading/trailing spaces
news_prep["text"] = news_prep["text"].str.strip()

In [8]:
# normalize repeated whitespace
news_prep["text"] = news_prep["text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [9]:
# check duplicates before removing just in case
news_prep["text"].duplicated().sum(), round(news_prep["text"].duplicated().mean() * 100, 2)

(np.int64(100), np.float64(4.49))

In [10]:
# remove duplicate articles
news_prep = news_prep.drop_duplicates(subset="text").copy()
news_prep.shape

(2125, 3)

In [11]:
# just to check for empty text after cleaning
(news_prep["text"].str.strip() == "").sum()

np.int64(0)

In [12]:
news_prep["num_words"] = news_prep["text"].str.split().str.len()
news_prep["num_words"].describe()

count    2125.000000
mean      384.115294
std       241.478609
min        89.000000
25%       245.000000
50%       331.000000
75%       471.000000
max      4432.000000
Name: num_words, dtype: float64

In [13]:
news_prep.loc[news_prep["num_words"].sort_values(ascending=False).index[:10], ["filename", "label", "num_words", "text"]]

,filename,label,num_words,text
1822,290.txt,politics,4432,Terror powers expose 'tyranny' The Lord Chance...
382,253.txt,entertainment,3482,Scissor Sisters triumph at Brits US band Sciss...
1636,380.txt,politics,3295,Minimum wage increased to Â£5.05 The minimum w...
1964,401.txt,tech,2969,Losing yourself in online gaming Online role p...
1795,293.txt,politics,2393,Kilroy launches 'Veritas' party Ex-BBC chat sh...
236,353.txt,entertainment,2391,Roundabout continues nostalgia trip The new bi...
298,256.txt,entertainment,2347,"Brits debate over 'urban' music Joss Stone, a ..."
1115,371.txt,sport,1662,All Black magic: New Zealand rugby Playing col...
2096,379.txt,tech,1528,Apple laptop is 'greatest gadget' The Apple Po...
1307,491.txt,sport,1383,Nadal puts Spain 2-0 up Result: Nadal 6-7 (6/8...


In [14]:
news_prep["text"] = news_prep["text"].apply(lambda x: " ".join(x.split()[:500]))

In [15]:
news_prep["num_words"] = news_prep["text"].str.split().str.len()
news_prep["num_words"].describe()

count    2125.000000
mean      343.674824
std       117.626148
min        89.000000
25%       245.000000
50%       331.000000
75%       471.000000
max       500.000000
Name: num_words, dtype: float64

In [16]:
news_prep["doc_type"] = "news"
news_final = news_prep[["filename", "text", "doc_type"]].copy()
news_final.head()

,filename,text,doc_type
0,289.txt,Musicians to tackle US red tape Musicians' gro...,news
1,262.txt,"U2's desire to be number one U2, who have won ...",news
2,276.txt,Rocker Doherty in on-stage fight Rock singer P...,news
3,060.txt,Snicket tops US box office chart The film adap...,news
4,074.txt,Ocean's Twelve raids box office Ocean's Twelve...,news


In [17]:
print(news_final.shape)
print(news_final["doc_type"].value_counts())
print((news_final["text"].str.strip() == "").sum())
print(news_final["text"].duplicated().sum())

(2125, 3)
doc_type
news    2125
Name: count, dtype: int64
0
2


## Preprocessing Emails

Based on the eda, there are many messy stuff:
- huge duplicate problem
- very long outliers
- some extremely short emails
- some missing Message
- oisy header-like structure

In [18]:
emails_prep = pd.read_csv("../data/raw/emails/enron_spam_data.csv")
emails_prep.shape

(33716, 5)

In [19]:
emails_prep = emails_prep.drop(columns=["Unnamed: 0"], errors="ignore")
emails_prep.columns.tolist()

['Subject', 'Message', 'Spam/Ham', 'Date']

In [20]:
emails_prep["Subject"] = emails_prep["Subject"].fillna("").astype(str)
emails_prep["Message"] = emails_prep["Message"].fillna("").astype(str)

emails_prep["text"] = (
    emails_prep["Subject"].str.strip() + " " + emails_prep["Message"].str.strip()
).str.strip()

emails_prep[["Subject", "Message", "text"]].head()

,Subject,Message,text
0,christmas tree farm pictures,,christmas tree farm pictures
1,"vastar resources , inc .","gary , production from the high island larger ...","vastar resources , inc . gary , production fro..."
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,calpine daily gas nomination - calpine daily g...
3,re : issue,fyi - see note below - already done .\nstella\...,re : issue fyi - see note below - already done...
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,meter 7268 nov allocation fyi .\n- - - - - - -...


In [21]:
emails_prep = emails_prep[["Subject", "Message", "Spam/Ham", "Date", "text"]].copy()
emails_prep.head()

,Subject,Message,Spam/Ham,Date,text
0,christmas tree farm pictures,,ham,1999-12-10,christmas tree farm pictures
1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13,"vastar resources , inc . gary , production fro..."
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14,calpine daily gas nomination - calpine daily g...
3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14,re : issue fyi - see note below - already done...
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14,meter 7268 nov allocation fyi .\n- - - - - - -...


In [22]:
emails_prep["text"] = emails_prep["text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [23]:
(emails_prep["text"].str.strip() == "").sum()

np.int64(0)

In [24]:
emails_prep = emails_prep[emails_prep["text"].str.strip() != ""].copy()
emails_prep.shape

(33716, 5)

In [25]:
emails_prep["num_words"] = emails_prep["text"].str.split().str.len()
emails_prep["num_words"].describe()

count    33716.000000
mean       364.762309
std        838.077937
min          1.000000
25%        165.000000
50%        190.000000
75%        365.000000
max      45450.000000
Name: num_words, dtype: float64

In [26]:
emails_prep.loc[emails_prep["num_words"].sort_values().index[:20], ["Subject", "Message", "num_words", "text"]]

,Subject,Message,num_words,text
1655,revised,,1,revised
29109,website,,1,website
7569,fyi,,1,fyi
2537,house pictures,,2,house pictures
3273,gymnastics pictures,,2,gymnastics pictures
7136,elena chilkina,hi,3,elena chilkina hi
13257,password - fast,,3,password - fast
13760,please see attached,,3,please see attached
6974,this,hurricane elana\n,3,this hurricane elana
0,christmas tree farm pictures,,4,christmas tree farm pictures


In [27]:
emails_prep = emails_prep[emails_prep["num_words"] >= 5].copy()
emails_prep.shape

(33700, 6)

In [28]:
emails_prep["text"].duplicated().sum(), round(emails_prep["text"].duplicated().mean() * 100, 2)

(np.int64(18125), np.float64(53.78))

In [29]:
emails_prep = emails_prep.drop_duplicates(subset="text").copy()
emails_prep.shape

(15575, 6)

In [30]:
emails_prep["num_words"] = emails_prep["text"].str.split().str.len()
emails_prep["num_words"].describe()

count    15575.000000
mean       347.581894
std       1085.865992
min          5.000000
25%         72.000000
50%        173.000000
75%        367.000000
max      45450.000000
Name: num_words, dtype: float64

In [31]:
emails_prep.loc[emails_prep["num_words"].sort_values(ascending=False).index[:10], ["Subject", "num_words", "text"]]

,Subject,num_words,text
14254,enron mentions,45450,enron mentions enron : a wake - up call the wa...
13901,enron mentions - 11 / 09 / 01 - 11 / 10 / 01,34912,enron mentions - 11 / 09 / 01 - 11 / 10 / 01 r...
14190,enron mentions,33039,enron mentions fall of a power giant : bailout...
14123,enron mentions - 11 / 24 / 01 - 11 / 25 / 01,28015,enron mentions - 11 / 24 / 01 - 11 / 25 / 01 a...
14146,enron mentions,27037,enron mentions enron and dynegy discuss plan t...
14167,enron mentions,26324,enron mentions don ' t bet it all on your empl...
13588,enron mentions,23343,enron mentions enron discusses credit line of ...
13867,enron mentions,22467,enron mentions enron slashes profits since 199...
13948,enron mentions,21372,enron mentions enron ' s lay turns down severa...
13845,enron mentions,20004,enron mentions dynegy is in talks on purchasin...


In [32]:
emails_prep = emails_prep[emails_prep["num_words"] <= 5000].copy()
emails_prep.shape

(15497, 6)

In [33]:
emails_prep["text"] = emails_prep["text"].apply(lambda x: " ".join(x.split()[:500]))

In [34]:
emails_prep["num_words"] = emails_prep["text"].str.split().str.len()
emails_prep["num_words"].describe()

count    15497.000000
mean       222.273279
std        168.166924
min          5.000000
25%         72.000000
50%        171.000000
75%        363.000000
max        500.000000
Name: num_words, dtype: float64

In [35]:
emails_prep["doc_type"] = "email"

In [36]:
emails_final = emails_prep[["text", "doc_type"]].copy()
emails_final.head()

,text,doc_type
1,"vastar resources , inc . gary , production fro...",email
2,calpine daily gas nomination - calpine daily g...,email
3,re : issue fyi - see note below - already done...,email
4,meter 7268 nov allocation fyi . - - - - - - - ...,email
5,"mcmullen gas for 11 / 99 jackie , since the in...",email


In [37]:
print(emails_final.shape)
print(emails_final["doc_type"].value_counts())
print((emails_final["text"].str.strip() == "").sum())
print(emails_final["text"].duplicated().sum())

(15497, 2)
doc_type
email    15497
Name: count, dtype: int64
0
2


## Preprocessing contracts

In [38]:
CONTRACTS_DIR = Path("../data/raw/contracts/CUAD_v1")
TXT_DIR = CONTRACTS_DIR / "full_contract_txt"
MASTER_CSV = CONTRACTS_DIR / "master_clauses.csv"

data = []

for file in TXT_DIR.glob("*.txt"):
    try:
        text = file.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = file.read_text(encoding="latin-1")
    
    data.append({
        "filename": file.name,
        "contract_name": file.stem,
        "text": text
    })

contracts = pd.DataFrame(data)
contracts.shape

(510, 3)

In [39]:
contracts_prep = contracts.copy()
contracts_prep.shape

(510, 3)

In [40]:
contracts_prep = contracts_prep[["filename", "contract_name", "text"]].copy()
contracts_prep.head()

,filename,contract_name,text
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...
3,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WE...,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WE...,1 ...
4,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEME...,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...


In [41]:
contracts_prep["text"] = contracts_prep["text"].fillna("").astype(str)

In [42]:
contracts_prep["text"] = contracts_prep["text"].str.strip()

In [43]:
contracts_prep["text"] = (
    contracts_prep["text"]
    .str.replace("&nbsp;", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [44]:
(contracts_prep["text"].str.strip() == "").sum()

np.int64(0)

In [45]:
contracts_prep["text"].duplicated().sum(), round(contracts_prep["text"].duplicated().mean() * 100, 2)

(np.int64(1), np.float64(0.2))

In [46]:
contracts_prep = contracts_prep.drop_duplicates(subset="text").copy()
contracts_prep.shape

(509, 3)

In [47]:
contracts_prep["num_words"] = contracts_prep["text"].str.split().str.len()
contracts_prep["num_words"].describe()

count      509.000000
mean      7872.998035
std       8371.268256
min        109.000000
25%       2472.000000
50%       5039.000000
75%      10211.000000
max      47733.000000
Name: num_words, dtype: float64

In [48]:
contracts_prep.loc[
    contracts_prep["num_words"].sort_values(ascending=False).index[:10],
    ["filename", "contract_name", "num_words", "text"]
]

,filename,contract_name,num_words,text
470,"GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...","GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...",47733,"Exhibit 10.6 Goosehead Insurance Agency, LLC F..."
178,PhasebioPharmaceuticalsInc_20200330_10-K_EX-10...,PhasebioPharmaceuticalsInc_20200330_10-K_EX-10...,45650,Exhibit 10.21 Certain information has been exc...
188,"CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...","CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...",45577,Exhibit 10.20 Pages where confidential treatme...
53,VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_EX...,VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_EX...,45143,Exhibit 10.4 FORM OF TRANSFER AND SERVICING AG...
255,"Array BioPharma Inc. - LICENSE, DEVELOPMENT AN...","Array BioPharma Inc. - LICENSE, DEVELOPMENT AN...",42742,[ * ] = Certain confidential information conta...
443,MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...,MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...,41962,Exhibit 10.14 OUTSOURCING AGREEMENT BETWEEN IN...
259,RevolutionMedicinesInc_20200117_S-1_EX-10.1_11...,RevolutionMedicinesInc_20200117_S-1_EX-10.1_11...,40936,Exhibit 10.1 [***] Certain information in this...
128,HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_...,HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_...,38518,Exhibit 10.18 Confidential EXECUTION COPY CERT...
142,GRANTIERRAENERGYINC_05_07_2012-EX-10.6-TRANSPO...,GRANTIERRAENERGYINC_05_07_2012-EX-10.6-TRANSPO...,35339,EXHIBIT 10.6 TRANSPORTATION CONTRACT SPECIFIC ...
139,AtnInternationalInc_20191108_10-Q_EX-10.1_1187...,AtnInternationalInc_20191108_10-Q_EX-10.1_1187...,34973,Exhibit 10.1 CERTAIN CONFIDENTIAL PORTIONS OF ...


In [49]:
contracts_prep = contracts_prep[contracts_prep["num_words"] <= 30000].copy()
contracts_prep.shape

(494, 4)

In [50]:
contracts_prep["text"] = contracts_prep["text"].apply(lambda x: " ".join(x.split()[:1500]))

In [51]:
contracts_prep["num_words"] = contracts_prep["text"].str.split().str.len()
contracts_prep["num_words"].describe()

count     494.000000
mean     1385.603239
std       315.071260
min       109.000000
25%      1500.000000
50%      1500.000000
75%      1500.000000
max      1500.000000
Name: num_words, dtype: float64

In [52]:
contracts_prep["doc_type"] = "contract"

In [53]:
contracts_final = contracts_prep[["text", "doc_type"]].copy()
contracts_final.head()

,text,doc_type
0,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...,contract
1,Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BEEN ...,contract
2,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...,contract
3,1 Exhibit 10.3 I-on. (LOGO) www.i-on.com 561.3...,contract
4,Exhibit 1 JOINT FILING AGREEMENT The undersign...,contract


In [54]:
print(contracts_final.shape)
print(contracts_final["doc_type"].value_counts())
print((contracts_final["text"].str.strip() == "").sum())
print(contracts_final["text"].duplicated().sum())

(494, 2)
doc_type
contract    494
Name: count, dtype: int64
0
0


## Preprocessing invoices

In [55]:
INVOICE_DIR = Path("../data/raw/invoices")
print("Invoice data dir:", INVOICE_DIR, "exists:", INVOICE_DIR.exists())
print("Files:", [f.name for f in INVOICE_DIR.iterdir() if not f.name.startswith(".")])

Invoice data dir: ../data/raw/invoices exists: True
Files: ['converted_invoice_dataset.csv']


In [56]:
# SROIE2019 is no longer available. Load the replacement English invoice dataset.
# "Input" column contains raw OCR-style invoice text; "Final_Output" is not used here.
_new_csv = Path("../data/raw/invoices/converted_invoice_dataset.csv")
_df_new  = pd.read_csv(_new_csv)

invoices = _df_new[["Input"]].rename(columns={"Input": "text"}).copy()
invoices["split"]  = "train"
invoices["doc_id"] = ["inv_" + str(i).zfill(4) for i in range(len(invoices))]

print(f"Loaded {len(invoices)} invoices from {_new_csv.name}")
print(invoices[["doc_id", "split"]].head(3))

Loaded 67 invoices from converted_invoice_dataset.csv
     doc_id  split
0  inv_0000  train
1  inv_0001  train
2  inv_0002  train


In [57]:
invoices_prep = invoices.copy()
invoices_prep.shape

(67, 3)

In [58]:
invoices_prep = invoices_prep[["split", "doc_id", "text"]].copy()
invoices_prep.head()

,split,doc_id,text
0,train,inv_0000,Cream and White Simple Minimalist Catering Ser...
1,train,inv_0001,Beige Elegant Professional Business Invoice\n\...
2,train,inv_0002,Black and White Clean Modern Invoice\n\nConsul...
3,train,inv_0003,Black and White Minimalist Business Invoice\n\...
4,train,inv_0004,White Minimalist Business Invoice\n\nSUBTOTALN...


In [59]:
invoices_prep["text"] = invoices_prep["text"].fillna("").astype(str)

In [60]:
invoices_prep["text"] = invoices_prep["text"].str.strip()

In [61]:
invoices_prep["text"] = invoices_prep["text"].str.replace(r"[ \t]+", " ", regex=True).str.strip()

In [62]:
(invoices_prep["text"].str.strip() == "").sum()

np.int64(0)

In [63]:
invoices_prep = invoices_prep[invoices_prep["text"].str.strip() != ""].copy()
invoices_prep.shape

(67, 3)

In [64]:
invoices_prep["text"].duplicated().sum(), round(invoices_prep["text"].duplicated().mean() * 100, 2)

(np.int64(3), np.float64(4.48))

In [65]:
invoices_prep = invoices_prep.drop_duplicates(subset="text").copy()
invoices_prep.shape

(64, 3)

In [66]:
invoices_prep["num_words"] = invoices_prep["text"].str.split().str.len()
invoices_prep["num_words"].describe()

count     64.000000
mean      84.421875
std       21.740026
min       51.000000
25%       70.000000
50%       82.000000
75%       92.250000
max      173.000000
Name: num_words, dtype: float64

In [67]:
invoices_prep.loc[
    invoices_prep["num_words"].sort_values().index[:10],
    ["doc_id", "split", "num_words", "text"]
]

,doc_id,split,num_words,text
43,inv_0043,train,51,Black Elegant Restaurant Invoice\n\nTo:\nISABE...
40,inv_0040,train,55,Modern Business Invoice\n\nBUSINESS NAME\nTYPE...
4,inv_0004,train,56,White Minimalist Business Invoice\n\nSUBTOTALN...
41,inv_0041,train,59,Purple Modern Trendy Calm Clean Customer Invoi...
47,inv_0047,train,59,Beige Simple & Modern Shape Invoice\n\nINVOICE...
50,inv_0050,train,59,Dark Aesthetic Minimal Invoice Template\n\nInv...
55,inv_0055,train,59,White And Beige Minimal Invoice\n\nQTY\nI N V ...
22,inv_0022,train,62,Red and White Simple Commercial Invoice\n\nMic...
38,inv_0038,train,62,Red and White Simple Commercial Invoice\n\nIte...
48,inv_0048,train,63,Beige Minimal With Leave Illustration Invoice\...


In [68]:
# Truncate to 500 words — matches the cap applied to news, emails, and contracts
invoices_prep["text"] = invoices_prep["text"].apply(lambda x: " ".join(x.split()[:500]))
invoices_prep["doc_type"] = "invoice"

In [69]:
invoices_final = invoices_prep[["text", "doc_type"]].copy()
invoices_final["is_synthetic"] = False
invoices_final.head()

,text,doc_type
0,Cream and White Simple Minimalist Catering Ser...,invoice
1,Beige Elegant Professional Business Invoice Wi...,invoice
2,Black and White Clean Modern Invoice Consultin...,invoice
3,Black and White Minimalist Business Invoice In...,invoice
4,White Minimalist Business Invoice SUBTOTALNO I...,invoice


In [70]:
# ── Synthetic invoice augmentation ───────────────────────────────────────────
# 64 real samples vs 15k+ emails = 242:1 ratio. Even with class_weight='balanced'
# the model can't learn robust invoice vocabulary from 64 examples.
# We add 500 diverse synthetic English invoices covering different styles,
# issuers, currencies, payment terms, and layouts — including the hardest case:
# invoices that read like emails ('Dear X, please find attached invoice...').

import random as _random
_random.seed(42)

_issuers = [
    'TechSolutions Ltd', 'Global Services Inc', 'Apex Consulting LLC',
    'Vertex Media Group', 'Horizon Logistics Corp', 'Pinnacle Design Studio',
    'Summit Software Ltd', 'Clarity Accounting Services', 'Nexus IT Solutions',
    'Blueprint Architecture Co', 'Orbit Marketing Agency', 'Delta Freight Inc',
    'Sterling Finance Corp', 'Cascade Holdings Inc', 'Redwood Group',
    'Ironclad Manufacturing', 'Sunrise Media Ltd', 'Granite Construction Inc',
]
_recipients = [
    'Acme Corporation', 'Blue Sky Retail Ltd', 'Metro Supplies Inc',
    'Westside Distribution Co', 'Northern Foods Ltd', 'Pacific Ventures LLC',
    'Salford & Co', 'Global Retail Corp', 'Helena Paquet Enterprises',
    'Margarita Perez Inc', 'Estelle Darcy Consulting', 'Kimberly Nguyen Ltd',
]
_items = [
    ('Professional consulting services', 3, 3200, 9600),
    ('Software licence annual subscription', 5, 1500, 7500),
    ('Website design and development', 1, 4850, 4850),
    ('IT support and maintenance', 12, 600, 7200),
    ('Freight and logistics services', 1, 2100, 2100),
    ('Marketing campaign management', 1, 3500, 3500),
    ('Accounting and audit services', 1, 2750, 2750),
    ('Legal advisory contract review', 4, 450, 1800),
    ('Cloud hosting services', 1, 980, 980),
    ('Training workshop delivery', 2, 1200, 2400),
    ('Graphic design services', 3, 850, 2550),
    ('Data analysis and reporting', 1, 5000, 5000),
    ('Equipment rental', 7, 300, 2100),
    ('Translation services', 10, 120, 1200),
    ('Security audit', 1, 6500, 6500),
]
_terms  = ['Net 30', 'Net 15', 'Net 60', 'Due on receipt', 'Net 45']
_curr   = ['$', 'EUR', 'GBP', 'USD']
_months = ['January','February','March','April','May','June',
           'July','August','September','October','November','December']

def _rand_date():
    y, m, d = _random.choice([2022,2023,2024,2025]), _random.randint(1,12), _random.randint(1,28)
    fmt = _random.randint(0, 4)
    if fmt == 0: return f'{d:02d}/{m:02d}/{y}'
    if fmt == 1: return f'{m:02d}/{d:02d}/{y}'
    if fmt == 2: return f'{y}-{m:02d}-{d:02d}'
    if fmt == 3: return f'{d} {_months[m-1]} {y}'
    return f'{_months[m-1]} {d}, {y}'

def _rand_inv_no():
    return f"{_random.choice(['INV','SI','INV-','TAX-INV'])}-{_random.randint(1000,9999)}"

def _make_invoice():
    issuer = _random.choice(_issuers); recipient = _random.choice(_recipients)
    inv_no = _rand_inv_no(); inv_date = _rand_date(); due_date = _rand_date()
    item = _random.choice(_items); term = _random.choice(_terms); curr = _random.choice(_curr)
    tax_pct = _random.choice([0, 5, 8, 10, 20])
    subtotal = item[3]; tax_amt = round(subtotal * tax_pct / 100, 2)
    total = round(subtotal + tax_amt, 2)
    style = _random.randint(0, 4)
    if style == 0:
        return (f'{issuer}\nTAX INVOICE\nInvoice No: {inv_no}\nInvoice Date: {inv_date}\n'
                f'Due Date: {due_date}\nPayment Terms: {term}\n\nBill To:\n{recipient}\n\n'
                f'Description: {item[0]}\nQty: {item[1]}  Unit Price: {curr} {item[2]:,.2f}  '
                f'Amount: {curr} {item[3]:,.2f}\n\n'
                f'Subtotal: {curr} {subtotal:,.2f}\nTax ({tax_pct}%): {curr} {tax_amt:.2f}\n'
                f'Total Amount Due: {curr} {total:,.2f}\n\nPlease remit payment by {due_date}.\n'
                f'Thank you for your business.')
    elif style == 1:
        return (f'INVOICE\nFrom: {issuer}\nTo: {recipient}\n'
                f'Invoice Number: {inv_no}    Date: {inv_date}\n'
                f'Payment Due: {due_date}    Terms: {term}\n\n'
                f'Item: {item[0]}\nQuantity: {item[1]}  Rate: {curr} {item[2]:,.2f}  '
                f'Total: {curr} {item[3]:,.2f}\n\n'
                f'Grand Total: {curr} {total:,.2f}\nBalance Due: {curr} {total:,.2f}\n\n'
                f'Payment terms: {term}. Late payments subject to 1.5% monthly interest.')
    elif style == 2:
        return (f'Invoice {inv_no}\nIssued by: {issuer}\nBilled to: {recipient}\n'
                f'Date of issue: {inv_date}  Due: {due_date}\n\n'
                f'Services rendered:\n  {item[0]}: {item[1]} x {curr} {item[2]:,.2f} = {curr} {item[3]:,.2f}\n\n'
                f'Subtotal {curr} {subtotal:,.2f}\nVAT {tax_pct}% {curr} {tax_amt:.2f}\n'
                f'Total {curr} {total:,.2f}\n\nPlease pay by {due_date}. Reference: {inv_no}.')
    elif style == 3:
        return (f'COMMERCIAL INVOICE\nSeller: {issuer}\nBuyer: {recipient}\n'
                f'PO Number: PO-{_random.randint(10000,99999)}\n'
                f'Invoice Date: {inv_date}\nPayment Due: {due_date}\nTerms: {term}\n\n'
                f'Description: {item[0]}\n'
                f'Unit price: {curr} {item[2]:,.2f}  Qty: {item[1]}  Line total: {curr} {item[3]:,.2f}\n\n'
                f'Invoice total: {curr} {total:,.2f}\nAmount due: {curr} {total:,.2f}\n\n'
                f'Wire transfer or cheque payable to {issuer}.\nInvoice ID: {inv_no}')
    else:  # Email-style invoice (hardest case)
        return (f'Dear {recipient},\n\n'
                f'Please find attached invoice {inv_no} for {item[0].lower()} '
                f'provided in the period ending {inv_date}.\n\n'
                f'Amount due: {curr} {total:,.2f} (incl. {tax_pct}% tax)\n'
                f'Payment terms: {term}\nDue date: {due_date}\n\n'
                f'Please remit payment to {issuer} at your earliest convenience.\n'
                f'Kind regards,\n{issuer}')

synth_inv = pd.DataFrame({
    'text':         [' '.join(_make_invoice().split()[:500]) for _ in range(500)],
    'doc_type':     'invoice',
    'is_synthetic': True,
})

invoices_final = pd.concat([invoices_final, synth_inv], ignore_index=True)
print(f'invoices_final after augmentation: {invoices_final.shape}')
print(f'  real:      {(~invoices_final["is_synthetic"]).sum()}')
print(f'  synthetic: {invoices_final["is_synthetic"].sum()}')


invoices_final after augmentation: (564, 2)
doc_type
invoice    564
Name: count, dtype: int64


In [71]:
# Summary check after augmentation
print('Sample synthetic invoice:')
print(invoices_final.iloc[-1]['text'][:300])


Sample synthetic invoice:
Dear Global Retail Corp, Please find attached invoice SI-4298 for graphic design services provided in the period ending 27 October 2024. Amount due: USD 2,805.00 (incl. 10% tax) Payment terms: Due on receipt Due date: 01/06/2022 Please remit payment to TechSolutions Ltd at your earliest convenience.


In [72]:
print(invoices_final.shape)
print(invoices_final["doc_type"].value_counts())
print((invoices_final["text"].str.strip() == "").sum())
print(invoices_final["text"].duplicated().sum())

(564, 2)
doc_type
invoice    564
Name: count, dtype: int64
0
0


In [73]:
print("news_final:", news_final.shape)
print("emails_final:", emails_final.shape)
print("contracts_final:", contracts_final.shape)
print("invoices_final:", invoices_final.shape)

news_final: (2125, 3)
emails_final: (15497, 2)
contracts_final: (494, 2)
invoices_final: (564, 2)


## final check

In [74]:
for name, df in {
    "news": news_final,
    "emails": emails_final,
    "contracts": contracts_final,
    "invoices": invoices_final,
}.items():
    print(f"\n{name}")
    print(df.columns.tolist())
    print("empty texts:", (df["text"].str.strip() == "").sum())
    print("duplicate texts:", df["text"].duplicated().sum())
    print("labels:", df["doc_type"].value_counts().to_dict())


news
['filename', 'text', 'doc_type']
empty texts: 0
duplicate texts: 2
labels: {'news': 2125}

emails
['text', 'doc_type']
empty texts: 0
duplicate texts: 2
labels: {'email': 15497}

contracts
['text', 'doc_type']
empty texts: 0
duplicate texts: 0
labels: {'contract': 494}

invoices
['text', 'doc_type']
empty texts: 0
duplicate texts: 0
labels: {'invoice': 564}


In [75]:
# fixes from the checks above
news_final = news_final.drop_duplicates(subset="text").copy()
news_final = news_final[["text", "doc_type"]].copy()
news_final.shape

emails_final = emails_final.drop_duplicates(subset="text").copy()
emails_final.shape

(15495, 2)

In [76]:
for name, df in {
    "news": news_final,
    "emails": emails_final,
    "contracts": contracts_final,
    "invoices": invoices_final,
}.items():
    print(f"\n{name}")
    print(df.columns.tolist())
    print("empty texts:", (df["text"].str.strip() == "").sum())
    print("duplicate texts:", df["text"].duplicated().sum())


news
['text', 'doc_type']
empty texts: 0
duplicate texts: 0

emails
['text', 'doc_type']
empty texts: 0
duplicate texts: 0

contracts
['text', 'doc_type']
empty texts: 0
duplicate texts: 0

invoices
['text', 'doc_type']
empty texts: 0
duplicate texts: 0


In [77]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

news_final.to_csv(PROCESSED_DIR / "news_preprocessed.csv", index=False)
emails_final.to_csv(PROCESSED_DIR / "emails_preprocessed.csv", index=False)
contracts_final.to_csv(PROCESSED_DIR / "contracts_preprocessed.csv", index=False)
invoices_final.to_csv(PROCESSED_DIR / "invoices_preprocessed.csv", index=False)

## Merging datasets

In [78]:
full_df = pd.concat(
    [news_final, emails_final, contracts_final, invoices_final],
    ignore_index=True
)

full_df.shape

(18676, 2)

In [79]:
full_df["doc_type"].value_counts()

doc_type
email       15495
news         2123
invoice       564
contract      494
Name: count, dtype: int64

In [80]:
(full_df["text"].str.strip() == "").sum()

np.int64(0)

In [81]:
full_df["text"].duplicated().sum()

np.int64(0)

In [82]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
full_df.to_csv("../data/processed/full_dataset_preprocessed.csv", index=False)